# adls_laqki — Colab Runner

Clones the `adls_laqki` repo, builds the conda environment, and makes all
project modules importable inside this Colab session.

---

| # | Section | When to run |
|---|---------|-------------|
| 0 | Configuration | Every session |
| 1 | Cache on Drive (optional) | Every session if caching |
| 2 | Install Miniforge | Once (or once per Drive cache path) |
| 3 | Clone / update repo | Every session |
| 4 | Create conda env | Once — takes ~10 min |
| 5 | Activate env | Every session |
| 6 | Verify | Every session |
| 7 | HuggingFace login | Every session if using gated models |
| 8 | Experiment | Your code |

Full guide: `docs/colab_runner.md`

## 0. Configuration

Edit the variables in this cell. All other cells read from these.

In [1]:
# ── USER CONFIG — edit this cell ──────────────────────────────────────────
REPO_URL  = "https://github.com/sc3321/adls_laqki.git"
BRANCH    = "qpe-dev-testing"              # branch or tag to check out
ENV_FILE  = "environment.yml"   # or "environment.kernels.yml" for CUDA kernel dev
REPO_DIR  = "/content/adls_laqki"
CONDA_DIR = "/opt/miniforge3"   # overridden below if Drive caching is enabled
HF_TOKEN  = ""                  # HuggingFace token; leave empty if not needed
# ──────────────────────────────────────────────────────────────────────────

import os
os.environ.update({
    "REPO_URL":  REPO_URL,
    "BRANCH":    BRANCH,
    "ENV_FILE":  ENV_FILE,
    "REPO_DIR":  REPO_DIR,
    "CONDA_DIR": CONDA_DIR,
})
print("Config loaded.")
print(f"  repo    : {REPO_URL}  [{BRANCH}]")
print(f"  env file: {ENV_FILE}")
print(f"  conda   : {CONDA_DIR}")

Config loaded.
  repo    : https://github.com/sc3321/adls_laqki.git  [qpe-dev-testing]
  env file: environment.yml
  conda   : /opt/miniforge3


## 1. (Optional) Cache Conda on Google Drive

Set `USE_DRIVE_CACHE = True` to install the conda environment into Drive.
The env then survives session disconnects and avoids the ~10 min reinstall
penalty on every reconnect.

Skip if you prefer a clean reinstall each session.

In [2]:
USE_DRIVE_CACHE = True  # set True to persist the conda env on Drive

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    CONDA_DIR = "/content/drive/MyDrive/colab_cache/adls_miniforge3"
    os.environ["CONDA_DIR"] = CONDA_DIR
    print(f"Drive mounted. Conda dir: {CONDA_DIR}")
else:
    print("Drive cache disabled — env is session-local at /opt/miniforge3.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Conda dir: /content/drive/MyDrive/colab_cache/adls_miniforge3


## 2. Install Miniforge

Installs Miniforge3 (conda + mamba) to `$CONDA_DIR`.
Automatically skipped if Miniforge is already present.

In [3]:
%%bash
if [ -f "${CONDA_DIR}/bin/conda" ]; then
    echo "Miniforge already at ${CONDA_DIR}"
    "${CONDA_DIR}/bin/conda" --version
    exit 0
fi

echo "Installing Miniforge to ${CONDA_DIR} ..."
wget -q "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh" -O /tmp/miniforge.sh
bash /tmp/miniforge.sh -b -p "${CONDA_DIR}"
rm /tmp/miniforge.sh
"${CONDA_DIR}/bin/conda" --version
echo "Miniforge installed."

Installing Miniforge to /content/drive/MyDrive/colab_cache/adls_miniforge3 ...
Miniforge installed.


ERROR: File or directory already exists: '/content/drive/MyDrive/colab_cache/adls_miniforge3'
If you want to update an existing installation, use the -u option.
bash: line 11: /content/drive/MyDrive/colab_cache/adls_miniforge3/bin/conda: No such file or directory


## 3. Clone or Update Repository

First run: clones the repo. Subsequent runs: checks out `$BRANCH` and pulls latest.

In [4]:
%%bash
set -e
if [ ! -d "${REPO_DIR}/.git" ]; then
    echo "Cloning ${REPO_URL} (branch: ${BRANCH}) ..."
    git clone "${REPO_URL}" "${REPO_DIR}"
    cd "${REPO_DIR}" && git checkout "${BRANCH}"
else
    echo "Repo found — pulling latest from origin/${BRANCH} ..."
    cd "${REPO_DIR}"
    git fetch origin
    git checkout "${BRANCH}"
    git pull origin "${BRANCH}"
fi
echo "Ready: $(cd ${REPO_DIR} && git log --oneline -1)"

Repo found — pulling latest from origin/qpe-dev-testing ...
Your branch is behind 'origin/qpe-dev-testing' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
Updating 3679721..b46fe0e
Fast-forward
 src/qpe/scorer/statistics.py | 5 ++++-
 1 file changed, 4 insertions(+), 1 deletion(-)
Ready: b46fe0e tell pyTorch that second-order gradients needed in advace


From https://github.com/sc3321/adls_laqki
   3679721..b46fe0e  qpe-dev-testing -> origin/qpe-dev-testing
Already on 'qpe-dev-testing'
From https://github.com/sc3321/adls_laqki
 * branch            qpe-dev-testing -> FETCH_HEAD


## 4. Create Conda Environment

Creates the `adls-quant-kernel` conda env from `$ENV_FILE`.
**First time only — takes 5–15 minutes.** Automatically skipped if the env exists.

| `ENV_FILE` | Use when |
|---|---|
| `environment.yml` | Standard QPE work: benchmarking, sensitivity scoring (default) |
| `environment.kernels.yml` | CUDA kernel dev — requires Colab instance with CUDA 12.1 |

In [5]:
%%bash
ENV_NAME="adls-quant-kernel"
MAMBA="${CONDA_DIR}/bin/mamba"

if "${CONDA_DIR}/bin/conda" env list 2>/dev/null | grep -q "^${ENV_NAME}"; then
    echo "Conda env '${ENV_NAME}' already exists — skipping creation."
    exit 0
fi

echo "Creating conda env '${ENV_NAME}' from ${REPO_DIR}/${ENV_FILE} ..."
echo "This may take 5-15 minutes ..."
cd "${REPO_DIR}"
"${MAMBA}" env create -f "${ENV_FILE}" 2>&1
echo "Done. Conda env '${ENV_NAME}' created."

Creating conda env 'adls-quant-kernel' from /content/adls_laqki/environment.yml ...
This may take 5-15 minutes ...
bash: line 12: /content/drive/MyDrive/colab_cache/adls_miniforge3/bin/mamba: No such file or directory
Done. Conda env 'adls-quant-kernel' created.


## 5. Activate Environment

Makes the conda env's packages importable in this notebook.
**Run this cell on every session restart** (after re-running section 0).

In [6]:
import sys, site

CONDA_ENV = f"{CONDA_DIR}/envs/adls-quant-kernel"
SITE_PKGS = f"{CONDA_ENV}/lib/python3.11/site-packages"
SRC_DIR   = f"{REPO_DIR}/src"

# addsitedir processes .pth files, activating editable installs such as mase
if SITE_PKGS not in sys.path:
    site.addsitedir(SITE_PKGS)

# Project source (qpe, backends, benchmark, ...)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Make conda env binaries available to subprocesses
os.environ["PATH"] = f"{CONDA_ENV}/bin:" + os.environ.get("PATH", "")

print("Environment activated.")
print(f"  conda env : {CONDA_ENV}")
print(f"  source dir: {SRC_DIR}")
print(f"  site-pkgs : {SITE_PKGS}")

Environment activated.
  conda env : /content/drive/MyDrive/colab_cache/adls_miniforge3/envs/adls-quant-kernel
  source dir: /content/adls_laqki/src
  site-pkgs : /content/drive/MyDrive/colab_cache/adls_miniforge3/envs/adls-quant-kernel/lib/python3.11/site-packages


## 6. Verify Setup

In [7]:
import subprocess, sys

# GPU
print("=== GPU ===")
r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"],
    capture_output=True, text=True,
)
print(r.stdout.strip() if r.returncode == 0 else "No GPU / nvidia-smi unavailable")

# Python
print(f"\n=== Python {sys.version} ===")

# Core packages
print("\n=== Packages ===")
for pkg in ["torch", "transformers", "pydantic", "numpy", "datasets", "accelerate"]:
    try:
        m = __import__(pkg)
        print(f"  OK   {pkg:<20} {getattr(m, '__version__', 'installed')}")
    except ImportError as e:
        print(f"  FAIL {pkg:<20} {e}")

# QPE modules
print("\n=== QPE ===")
try:
    from qpe.solver.types import Precision, SolverInput, SolverOutput
    print("  OK   qpe.solver.types")
except ImportError as e:
    print(f"  FAIL qpe.solver.types: {e}")

# PyTorch CUDA
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"\n  torch.cuda.is_available() = {cuda_ok}")
    if cuda_ok:
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
except Exception as e:
    print(f"\n  torch CUDA check failed: {e}")

=== GPU ===
NVIDIA L4, 23034, 580.82.07

=== Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0] ===

=== Packages ===
  OK   torch                2.10.0+cu128
  OK   transformers         5.0.0
  OK   pydantic             2.12.3
  OK   numpy                2.0.2


  OK   datasets             4.0.0
  OK   accelerate           1.12.0

=== QPE ===
  FAIL qpe.solver.types: No module named 'qpe.solver.types'

  torch.cuda.is_available() = True
  GPU: NVIDIA L4


## 7. HuggingFace Login

Only needed for gated models (LLaMA, Mistral, Gemma, etc.).
Set `HF_TOKEN` in section 0, then run this cell.

In [8]:
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to HuggingFace Hub.")
else:
    print("HF_TOKEN not set — public models only.")

HF_TOKEN not set — public models only.


## 8. BERT Mixed-Precision Quantization Pipeline

Runs the full QPE pipeline on `textattack/bert-base-uncased-SST-2` using `ILPQualityMinimizer`.

| Stage | What happens |
|-------|-------------|
| Score | GuidedQuant saliency per layer (50 Hutchinson samples) |
| Profile | Latency + memory at FP16 / INT8 / INT4 per layer |
| Solve | ILP minimises CE-loss increase under `MEMORY_BUDGET_GB` |
| Validate | Accuracy + proxy perplexity on GLUE SST-2 validation split |
| Export | `bert_qpe_output/quantization_config.json` |

Adjust `MEMORY_BUDGET_GB` to trade accuracy against memory footprint.

In [9]:
MODEL_ID          = "textattack/bert-base-uncased-SST-2"
MEMORY_BUDGET_GB  = 0.05   # BERT-base FP16 weights ~440 MB; 0.3 forces INT8/INT4 on most layers
OUTPUT_DIR        = "/content/bert_qpe_output"
GPU_SPEC_NAME     = None  # None = auto-detect via pynvml

In [10]:
import time
from transformers import AutoTokenizer

from qpe.calibration.manager import CalibrationDataManager
from qpe.calibration.models import CalibrationConfig
from qpe.export.exporter import ConfigurationExporter
from qpe.pipeline import Pipeline, PipelineConfig
from qpe.profiler.gpu_specs import GPU_REGISTRY, detect_gpu
from qpe.profiler.layer_profiler import LayerProfiler
from qpe.scorer.hessian import HessianTraceScorer, HessianTraceScorerConfig
from qpe.solver.config import QualityMinimizerConfig
from qpe.utils.types import Precision
from qpe.validation.bert_engine import BERTValidationEngine
from qpe.validation.config import ValidationConfig

gpu_spec = GPU_REGISTRY[GPU_SPEC_NAME] if GPU_SPEC_NAME else detect_gpu()
print(f"GPU: {gpu_spec.name} ({gpu_spec.memory_gb:.1f} GB)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

calibration_config = CalibrationConfig(
    num_samples=256,
    sequence_length=128,
    datasets=["sst2"],
    validation_split=0.2,
)
calibration_manager = CalibrationDataManager(calibration_config, tokenizer)

scorer = HessianTraceScorer(HessianTraceScorerConfig(
    saliency_mode="guided",
    compute_hessian_trace=True,
    n_hutchinson_samples=50,
))

profiler = LayerProfiler(gpu_spec=gpu_spec, num_warmup=10, num_measurements=50, seq_len=128)

validation_config = ValidationConfig(
    perplexity_dataset="glue/sst2",
    max_perplexity_increase_pct=5.0,
    benchmark_tasks=["sst2_accuracy"],
    kl_threshold_per_layer=0.1,
)
validator = BERTValidationEngine(
    config=validation_config,
    val_dataloader=calibration_manager.get_validation_dataloader(batch_size=32),
    gpu_spec=gpu_spec,
)

solver_config = QualityMinimizerConfig(
    memory_budget_gb=MEMORY_BUDGET_GB,
    precision_candidates=[Precision.FP16, Precision.W8A8_INT8, Precision.W4A16],
    pinned_layers={
        "bert.pooler.dense": Precision.FP16.value,
        "classifier": Precision.FP16.value,
    },
)

pipeline = Pipeline(
    calibration_manager=calibration_manager,
    scorer=scorer,
    profiler=profiler,
    validator=validator,
    exporter=ConfigurationExporter(),
    config=PipelineConfig(
        model_id=MODEL_ID,
        calibration=calibration_config,
        validation=validation_config,
        solver=solver_config,
        export_target="pytorch",
        output_dir=OUTPUT_DIR,
        task="sequence_classification",
        gpu_spec_name=GPU_SPEC_NAME,
        target_batch_size=1,
        max_feedback_iterations=3,
    ),
)
print("Pipeline ready.")

GPU: NVIDIA L4 (22.5 GB)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Pipeline ready.


In [11]:
t0 = time.perf_counter()
result = pipeline.run()
print(f"Done in {time.perf_counter() - t0:.1f}s")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Done in 222.7s


In [12]:
from collections import Counter

so = result.solver_output
fb = result.validation

print("=== QPE BERT Results ===")
print(f"Solver status  : {so.solver_status}")
print(f"Avg bitwidth   : {so.average_bitwidth:.2f} bits")
print(f"Weight memory  : {so.total_memory_bytes / 1e6:.1f} MB")
print(f"Solve time     : {so.solve_time_seconds:.2f}s")
print(f"Iterations     : {result.iterations_used}")
print(f"\nAccuracy FP16  : {fb.fp16_benchmark_scores.get('sst2_accuracy', 0):.4f}")
print(f"Accuracy quant : {fb.benchmark_scores.get('sst2_accuracy', 0):.4f}")
print(f"PPL increase   : {fb.perplexity_increase_pct:.2f}%")
print(f"Passed         : {fb.passed}")

dist = Counter(so.to_assignment_dict().values())
print(f"\nPrecision distribution:")
for prec, count in sorted(dist.items()):
    print(f"  {prec:<15} {count} layers")

print(f"\nOutput: {result.export.output_path}")

=== QPE BERT Results ===
Solver status  : optimal
Avg bitwidth   : 16.00 bits
Weight memory  : 170.0 MB
Solve time     : 0.05s
Iterations     : 1

Accuracy FP16  : 0.9243
Accuracy quant : 0.9243
PPL increase   : 0.00%
Passed         : True

Precision distribution:
  FP16            72 layers

Output: /content/bert_qpe_output


In [ ]:
import random
from qpe.solver.models import LayerAssignment, SolverOutput

N_TRIALS = 5
PINNED_LAYERS = {"bert.pooler.dense", "classifier"}
MEM_SCALE = {"FP16": 1.0, "W8A8_INT8": 0.5, "W4A16": 0.25}

so = result.solver_output
free_assignments = [a for a in so.assignments if a.layer_name not in PINNED_LAYERS]
pinned_assignments = [a for a in so.assignments if a.layer_name in PINNED_LAYERS]
precision_pool = [a.assigned_precision for a in free_assignments]
pinned_mem = sum(a.estimated_memory_bytes for a in pinned_assignments)
optimizer_mem = sum(a.estimated_memory_bytes for a in so.assignments)

def fp16_mem(a):
    """Recover FP16-equivalent memory from an assignment."""
    return a.estimated_memory_bytes / MEM_SCALE[a.assigned_precision.value]

def make_assignments(free_originals, precisions):
    return [
        LayerAssignment(
            layer_name=a.layer_name,
            assigned_precision=p,
            estimated_quality_cost=a.estimated_quality_cost,
            estimated_memory_bytes=a.estimated_memory_bytes,
            estimated_latency_us=a.estimated_latency_us,
        )
        for a, p in zip(free_originals, precisions)
    ] + pinned_assignments

def make_solver_output(assignments):
    return SolverOutput(
        assignments=assignments,
        total_estimated_quality_cost=so.total_estimated_quality_cost,
        total_memory_bytes=so.total_memory_bytes,
        total_memory_with_kv_bytes=so.total_memory_with_kv_bytes,
        total_latency_us=so.total_latency_us,
        average_bitwidth=so.average_bitwidth,
        solver_name="baseline",
        solver_status="baseline",
        solve_time_seconds=0.0,
        formulation_used="baseline",
        kv_cache_dtype=so.kv_cache_dtype,
        sensitivity_ranking=so.sensitivity_ranking,
    )

def actual_memory(free_with_new_precision):
    return pinned_mem + sum(
        fp16_mem(orig) * MEM_SCALE[a.assigned_precision.value]
        for orig, a in zip(free_assignments, free_with_new_precision)
    )

def evaluate(assignments, label):
    mem = actual_memory(assignments)
    fb = pipeline.validator.validate(None, pipeline.config.model_id, make_solver_output(assignments), stage="full")
    acc = fb.benchmark_scores.get("sst2_accuracy", 0)
    ppl = fb.perplexity_increase_pct
    print(f"  {label:<30} accuracy={acc:.4f}  PPL={ppl:+.2f}%  memory={mem/1e6:.1f} MB")
    return acc, ppl, mem

from qpe.utils.types import Precision
opt_acc = result.validation.benchmark_scores.get("sst2_accuracy", 0)
print(f"{'='*70}")
print(f"  {'Optimizer':<30} accuracy={opt_acc:.4f}  PPL={result.validation.perplexity_increase_pct:+.2f}%  memory={optimizer_mem/1e6:.1f} MB")
print(f"{'='*70}")

results = {}

# --- Baseline 1: Random (N trials) ---
print(f"\n[1] Random ({N_TRIALS} trials)")
rand_accs, rand_mems = [], []
for trial in range(N_TRIALS):
    shuffled = random.sample(precision_pool, len(precision_pool))
    assignments = make_assignments(free_assignments, shuffled)
    acc, ppl, mem = evaluate(assignments, f"Trial {trial+1}")
    rand_accs.append(acc)
    rand_mems.append(mem)
results["random"] = (sum(rand_accs)/len(rand_accs), sum(rand_mems)/len(rand_mems))

# --- Baseline 2: All W4A16 ---
print(f"\n[2] All W4A16")
assignments = make_assignments(free_assignments, [Precision.W4A16] * len(free_assignments))
acc, ppl, mem = evaluate(assignments, "All W4A16")
results["all_w4"] = (acc, mem)

# --- Baseline 3: All W8A8_INT8 ---
print(f"\n[3] All W8A8_INT8")
assignments = make_assignments(free_assignments, [Precision.W8A8_INT8] * len(free_assignments))
acc, ppl, mem = evaluate(assignments, "All W8A8_INT8")
results["all_w8"] = (acc, mem)

# --- Baseline 4: Greedy by size (W4A16 on largest layers until budget hit) ---
print(f"\n[4] Greedy by size")
budget_bytes = MEMORY_BUDGET_GB * 1e9
sorted_by_size = sorted(free_assignments, key=lambda a: fp16_mem(a), reverse=True)
greedy_prec = {a.layer_name: Precision.W8A8_INT8 for a in free_assignments}
running_mem = pinned_mem + sum(fp16_mem(a) * MEM_SCALE["W8A8_INT8"] for a in free_assignments)
for a in sorted_by_size:
    saving = fp16_mem(a) * (MEM_SCALE["W8A8_INT8"] - MEM_SCALE["W4A16"])
    if running_mem - saving >= 0:  # always apply — greedy takes as much W4A16 as fits
        greedy_prec[a.layer_name] = Precision.W4A16
        running_mem -= saving
        if running_mem <= budget_bytes:
            break
greedy_precisions = [greedy_prec[a.layer_name] for a in free_assignments]
assignments = make_assignments(free_assignments, greedy_precisions)
acc, ppl, mem = evaluate(assignments, "Greedy by size")
results["greedy"] = (acc, mem)

# --- Summary ---
print(f"\n{'='*70}")
print(f"{'Baseline':<30} {'Accuracy':>10} {'vs Opt':>8} {'Memory':>10}")
print(f"{'-'*70}")
print(f"{'Optimizer':<30} {opt_acc:>10.4f} {'---':>8} {optimizer_mem/1e6:>9.1f}MB")
labels = {"random": "Random (avg)", "all_w4": "All W4A16", "all_w8": "All W8A8_INT8", "greedy": "Greedy by size"}
for key, label in labels.items():
    acc, mem = results[key]
    print(f"{label:<30} {acc:>10.4f} {opt_acc-acc:>+8.4f} {mem/1e6:>9.1f}MB")


In [ ]:
# Memory footprint comparison: optimizer vs random baseline
from qpe.utils.types import Precision

PINNED_LAYERS = {"bert.pooler.dense", "classifier"}
so = result.solver_output

free = [a for a in so.assignments if a.layer_name not in PINNED_LAYERS]
pinned = [a for a in so.assignments if a.layer_name in PINNED_LAYERS]

# Memory cost relative to FP16 per precision
MEM_SCALE = {"FP16": 1.0, "W8A8_INT8": 0.5, "W4A16": 0.25}

def layer_mem_at(a, precision_str):
    """Estimate a layer's memory at a given precision, using its assigned mem as FP16 reference."""
    assigned_scale = MEM_SCALE[a.assigned_precision.value]
    fp16_mem = a.estimated_memory_bytes / assigned_scale
    return fp16_mem * MEM_SCALE[precision_str]

# Count each precision
from collections import Counter
dist = Counter(a.assigned_precision.value for a in free)
print("Precision split (free layers):", dict(dist))
print()

# Optimizer memory
pinned_mem = sum(a.estimated_memory_bytes for a in pinned)
optimizer_mem = sum(a.estimated_memory_bytes for a in so.assignments)

# Expected memory under random assignment with same precision counts
# For each free layer, expected mem = weighted avg across precision pool
total_free = len(free)
expected_free_mem = sum(
    sum(count * layer_mem_at(a, prec) for prec, count in dist.items()) / total_free
    for a in free
)
expected_random_mem = expected_free_mem + pinned_mem

print(f"Budget                    : {MEMORY_BUDGET_GB*1000:.1f} MB")
print(f"Optimizer memory          : {optimizer_mem/1e6:.1f} MB")
print(f"Expected random memory    : {expected_random_mem/1e6:.1f} MB")
print(f"Optimizer vs random       : {(optimizer_mem - expected_random_mem)/1e6:+.1f} MB")
print()
print("Interpretation:")
print("  +ve = optimizer uses MORE memory than random (puts expensive precision on large layers)")
print("  -ve = optimizer uses LESS memory than random (puts cheap precision on large layers)")
print()

# Per-precision avg layer size
for prec in dist:
    layers = [a for a in free if a.assigned_precision.value == prec]
    fp16_sizes = [layer_mem_at(a, "FP16") for a in layers]
    print(f"  Avg FP16-equiv size of {prec:12s} layers: {sum(fp16_sizes)/len(fp16_sizes)/1e6:.3f} MB")


## 9. Random Baseline

Shuffles the optimizer's precision assignments randomly across the same layers,
keeping the **exact same precision counts** (e.g. 47×W4A16, 25×W8A8_INT8).
Runs N trials and compares average accuracy against the optimizer.

In [13]:
from google.colab import runtime
runtime.unassign()